In [0]:
%run ../utils

In [0]:
import os
import json
import requests
from pydantic import BaseModel, Field

In [0]:
dbutils.widgets.text("base_llm_model", "databricks-meta-llama-3-3-70b-instruct")
base_llm_model = dbutils.widgets.get("base_llm_model")
print("Base model:", base_llm_model)
#ananya-rac-gpt4o-ep
#databricks-llama-4-maverick

In [0]:
client = get_client()

In [0]:
"""
docs: https://platform.openai.com/docs/guides/function-calling
"""

In [0]:
# --------------------------------------------------------------
# Define the tool (function) that we want to call
# --------------------------------------------------------------

def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

In [0]:
# --------------------------------------------------------------
# Step 1: Call model with get_weather tool defined
# --------------------------------------------------------------

tools = [
  {
    "type" : "function",
    "function" : {
      "name" : "get_weather",
      "description" : "Get current temperature for provided coordinates in celcius.",
      "parameters" : {
        "type" : "object",
        "properties" : {
          "latitude" : {"type" : "number"},
          "longitude" : {"type" : "number"}
        },
        "required" : ["latitude", "longitude"],
        "additionalProperties" : False
      },
      "strict" : True,
    },
  }
]

system_prompt = "You are a helpful weather assistant"

messages = [
  {"role" : "system", "content" : system_prompt},
  {"role" : "user", "content" : "What's the weather like in Paris today?"}
]

completion = get_completion_create(
  oai_client= client,
  model=base_llm_model,
  messages=messages,
  tools=tools
)

In [0]:
# --------------------------------------------------------------
# Step 2: Model decides to call function(s)
# --------------------------------------------------------------
completion.choices[0]

In [0]:
completion.model_dump()

In [0]:
completion.choices[0].message

In [0]:
completion.choices[0].message.tool_calls

In [0]:
# --------------------------------------------------------------
# Step 3: Execute get_weather function
# --------------------------------------------------------------

def call_function(name, args):
  if name == "get_weather":
    return get_weather(**args)
  
for tool_call in completion.choices[0].message.tool_calls:
  name = tool_call.function.name
  args = json.loads(tool_call.function.arguments)
  messages.append(completion.choices[0].message)

  result = call_function(name, args)
  messages.append(
    {"role" : "tool", "tool_call_id" : tool_call.id, "content" : json.dumps(result)}
  )

In [0]:
messages

In [0]:
# --------------------------------------------------------------
# Step 4: Supply result and call model again
# --------------------------------------------------------------

class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )

completion_2 = get_completion_parse(
  oai_client=client,
  model=base_llm_model,
  messages=messages,
  tools=tools,
  response_format=WeatherResponse
)

In [0]:
# --------------------------------------------------------------
# Step 5: Check model response
# --------------------------------------------------------------

final_response = completion_2.choices[0].message.parsed
print("temp: ", final_response.temperature)
print("response: ", final_response.response)